In [1]:
# Import Packages
import pandas as pd
import re

# Import from local files
from data_cleaning import DataCleaner
from extraction import extract_data
from entity_resolution import EntityResolver

resolver = EntityResolver()

# Set pandas display options
pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)


# ----------------------------------------------------------
# Extract
# ----------------------------------------------------------

extracted_df = extract_data(
    "2025 Roll - Parcels Export.xlsx",
    "Only Delinquent Parcels"
)


# ----------------------------------------------------------
# Clean
# ----------------------------------------------------------

cleaner = DataCleaner()

clean_df = cleaner.clean_data(
    extracted_df
)


# ----------------------------------------------------------
# Entity Resolution
# ----------------------------------------------------------

(
    entity_occurrences,
    entities,
    parcel_entity_roles,
    association_evidence,
    entity_associations
) = EntityResolver().build(
    clean_df
)

# Create audit dataframe.
resolution_audit = (
    entity_occurrences
    .merge(
        entities[
            [
                "entity_id",
                "canonical_name",
                "canonical_address",
                "entity_type",
                "occurrence_count"
            ]
        ],
        on="entity_id",
        how="left",
        validate="many_to_one",
        suffixes=("_source", "_final")
    )
)

resolution_audit.head()

,occurrence_id,parcel_number,entity_name,associated_address,role,source_entity_name,source_column,source_occurrence_id,entity_type_source,joint_parse_rule,joint_parse_confidence,joint_expanded,joint_party_index,joint_party_count,organization_type,organization_classification_confidence,address_source,entity_id,canonical_name,canonical_address,entity_type_final,occurrence_count
0,1,01-01-201-013,POLOMCAK CLAUDIA,"31029 30TH ST, PAW PAW, MI 49079",Owner,POLOMCAK CLAUDIA,owner_name,1,Person,,,False,NaN,NaN,NaN,NaN,owner_address,3342,POLOMCAK CLAUDIA,"31029 30TH ST, PAW PAW, MI 49079",Person,2
1,2,01-01-251-013,CLAWSON DORRINDA &,"10591 N 12TH ST, PLAINWELL, MI 49080",Owner,CLAWSON DORRINDA &,owner_name,2,Joint,,,False,NaN,NaN,NaN,NaN,owner_address,927,CLAWSON DORRINDA &,"10591 N 12TH ST, PLAINWELL, MI 49080",Joint,2
2,3,01-01-251-050,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Owner,CLAWSON DORRINDA L,owner_name,3,Person,,,False,NaN,NaN,NaN,NaN,owner_address,928,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Person,6
3,4,01-01-251-060,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Owner,CLAWSON DORRINDA L,owner_name,4,Person,,,False,NaN,NaN,NaN,NaN,owner_address,928,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Person,6
4,5,01-01-301-050,SPIEKERMANN TIM & JUDY,"10238 N 10TH ST, PLAINWELL, MI 49080",Owner,SPIEKERMANN TIM & JUDY,owner_name,5,Joint,,,False,NaN,NaN,NaN,NaN,owner_address,3973,SPIEKERMANN TIM & JUDY,"10238 N 10TH ST, PLAINWELL, MI 49080",Joint,2


## Entities with the highest occurrence_count

In [2]:
entities.sort_values(
    "occurrence_count",
    ascending=False
).head(10)

,entity_id,canonical_name,entity_type,canonical_address,organization_type,organization_classification_confidence,occurrence_count
2188,2189,KAYLOR HOLDINGS LLC,Organization,"1405 HOLIDAY LN, PORTAGE, MI 49024",LLC,High,62
603,604,"BROOKS, SAMUEL C",Person,"804 CHARLOTTE AVE, KALAMAZOO, MI 49048",NaN,NaN,44
4002,4003,STATE WIDE RENTALS LLC,Organization,"606 BRYANT ST, KALAMAZOO, MI 49001",LLC,High,25
20,21,"19 PROPS, LLC",Organization,"200 W MICHIGAN AVE STE 201, KALAMAZOO, MI 49007",LLC,High,21
443,444,"BISHOP, BRADLEY J",Person,"PO BOX 312, PORTAGE, MI 49081",NaN,NaN,19
1846,1847,"HERTEL, JOSEPH B",Person,"2326 S ROSE ST, KALAMAZOO, MI 49001",NaN,NaN,18
4431,4432,WALLY BROWN LLC,Organization,"563 TREASURE ISLAND DR, MATTAWAN, MI 49071",LLC,High,18
4709,4710,"YOUNG, CAVEL J",Person,"2111 N DRAKE RD, KALAMAZOO, MI 49006",NaN,NaN,18
1571,1572,GIBSON ROBERT M & MARTHA M,Joint,"8713 S 38TH ST, SCOTTS, MI 49088",NaN,NaN,18
3917,3918,"SMITH, SEAN A",Person,"5093 W GH AVE, KALAMAZOO, MI 49009",NaN,NaN,17


## Entities with canonical names containing a middle name or middle initial

In [3]:
SUFFIXES = {
    "JR",
    "SR",
    "JUNIOR",
    "SENIOR",
    "II",
    "III",
    "IV",
}


def get_middle_tokens(name):
    """
    Return possible middle-name tokens from a person name.

    Supports both:
        SMITH, JOHN A
        SMITH JOHN A

    This is for manual inspection only, not entity-resolution logic.
    """

    if pd.isna(name):
        return tuple()

    name = str(name).upper().strip()
    name = name.replace(".", "")

    # Skip obvious joint/compound records
    if (
        "&" in name
        or "/" in name
        or re.search(r"\bAND\b", name)
    ):
        return tuple()

    if "," in name:

        _, given_part = name.split(",", 1)
        tokens = given_part.strip().split()

        if tokens and tokens[-1] in SUFFIXES:
            tokens = tokens[:-1]

        # FIRST [MIDDLE...]
        return tuple(tokens[1:])

    else:

        tokens = name.split()

        if tokens and tokens[-1] in SUFFIXES:
            tokens = tokens[:-1]

        # Assume LAST FIRST [MIDDLE...]
        if len(tokens) < 3:
            return tuple()

        return tuple(tokens[2:])

people_with_middle_names = (
    entities[
        entities["entity_type"] == "Person"
    ]
    .copy()
)

people_with_middle_names["middle_tokens"] = (
    people_with_middle_names[
        "canonical_name"
    ]
    .apply(get_middle_tokens)
)

people_with_middle_names = (
    people_with_middle_names[
        people_with_middle_names[
            "middle_tokens"
        ].apply(len) > 0
    ]
    .sort_values(
        "occurrence_count",
        ascending=False
    )
)

people_with_middle_names[
    [
        "entity_id",
        "canonical_name",
        "middle_tokens",
        "canonical_address",
        "occurrence_count"
    ]
]

,entity_id,canonical_name,middle_tokens,canonical_address,occurrence_count
603,604,"BROOKS, SAMUEL C","(C,)","804 CHARLOTTE AVE, KALAMAZOO, MI 49048",44
443,444,"BISHOP, BRADLEY J","(J,)","PO BOX 312, PORTAGE, MI 49081",19
4709,4710,"YOUNG, CAVEL J","(J,)","2111 N DRAKE RD, KALAMAZOO, MI 49006",18
1846,1847,"HERTEL, JOSEPH B","(B,)","2326 S ROSE ST, KALAMAZOO, MI 49001",18
3917,3918,"SMITH, SEAN A","(A,)","5093 W GH AVE, KALAMAZOO, MI 49009",17
...,...,...,...,...,...
4652,4653,"WOLFE, LANCE L","(L,)","11367 3 MILE RD, PLAINWELL, MI 49080",1
4651,4652,WOLFE WILLIAM C,"(C,)","PO BOX 623, COMSTOCK, MI 49041",1
4650,4651,WOLFE WILLIAM C,"(C,)","1355 N 37TH ST, GALESBURG, MI 49053-9762",1
4614,4615,"WILLSON, ALAN R","(R,)","222 E ALCOTT ST, KALAMAZOO, MI 49001",1


- Some names have commas while others don't
    - For example, KAVANAUGH, JAMES J vs. CLAWSON DORRINDA L
        - Note that both had their middle initial listed correctly under middle_tokens
- The canonical name FRITH, TERRANCE JEROME A produced the middle_token (JEROME, A)
- The canonical name BROWN, ROBERT WENDELL, JR. produced the middle_token (WENDELL,,)
    - Note that there are two commas rather than the usual one
- GREATER TRUE WORSHIP COGIC produced (WORSHIP, COGIC)
- BLACKSTAR STABILITY REVITALIZATION produced (REVITALIZATION,)
- EDWARDS, KARRIE; EDWARDS, KENTON produced (EDWARDS,, KENTON)

Unrelated case:
There are two entities with the name WIDGER JAMES LEROY, one with the address 214 MARSHALL ST, ALLEGAN, MI 49010 and the other with the address 214 MARSHALL ST, ALLEGAN, MI 49010. Because they have the exact same name and their address, with the removal of ST from the first case, are the same - they should clearly be merged.

### Distinguishing middle initials from full middle names

In [4]:
def middle_type(tokens):

    if not tokens:
        return None

    if all(len(token) == 1 for token in tokens):
        return "Initial"

    return "Full/Other"


people_with_middle_names["middle_type"] = (
    people_with_middle_names[
        "middle_tokens"
    ]
    .apply(middle_type)
)

people_with_middle_names[
    "middle_type"
].value_counts()

middle_type
Initial       883
Full/Other    400
Name: count, dtype: int64

Out of 3,955 unique entity names, 1,240 show as containing a middle name or initial.

In [5]:
797 + 443

1240

## Entities the previously had a blank address

In [6]:
blank_address_cases = (
    resolution_audit
    .groupby(
        [
            "entity_id",
            "canonical_name",
            "canonical_address"
        ],
        dropna=False
    )
    .agg(
        had_blank_address=(
            "associated_address",
            lambda x: (
                x.fillna("")
                .str.strip()
                .eq("")
                .any()
            )
        ),
        source_addresses=(
            "associated_address",
            lambda x: tuple(
                sorted(
                    {
                        address
                        for address in x.fillna("")
                        if address != ""
                    }
                )
            )
        ),
        source_names=(
            "entity_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        occurrence_count=(
            "occurrence_id",
            "size"
        )
    )
    .reset_index()
)

blank_address_cases = (
    blank_address_cases[
        blank_address_cases["had_blank_address"]
        &
        (
            blank_address_cases[
                "canonical_address"
            ] != ""
        )
    ]
)

print(blank_address_cases.shape)
blank_address_cases.head(50)

(1670, 7)


,entity_id,canonical_name,canonical_address,had_blank_address,source_addresses,source_names,occurrence_count
15,16,1733 W. MAIN ST LLC,"1733 1/2 W MAIN ST, KALAMAZOO, MI 49006",True,"(1733 1/2 W MAIN ST, KALAMAZOO, MI 49006,)","(1733 W. MAIN ST LLC,)",2
19,20,19 PROPS LLC,"200 W MICHIGAN AVE STE 201, KALAMAZOO, MI 49007-",True,"(200 W MICHIGAN AVE STE 201, KALAMAZOO, MI 490...","(19 PROPS LLC,)",2
21,22,"1930 WEST MAIN, LLC","3260 NILES RD STE B, SAINT JOSEPH, MI 49085",True,"(3260 NILES RD STE B, SAINT JOSEPH, MI 49085,)","(1930 WEST MAIN, LLC,)",2
24,25,2219 HUMPHREY LLC,"100 W MICHIGAN AVE STE 200, KALAMAZOO, MI 49007",True,"(100 W MICHIGAN AVE STE 200, KALAMAZOO, MI 490...","(2219 HUMPHREY LLC,)",2
26,27,232 GRAND LLC,"11670 RAMSEYER DR, PLAINWELL, MI 49080",True,"(11670 RAMSEYER DR, PLAINWELL, MI 49080,)","(232 GRAND LLC,)",2
28,29,26TH ST PROPERTIES LLC,"2915 E GRAND RIVER AVE, HOWELL, MI 48843",True,"(2915 E GRAND RIVER AVE, HOWELL, MI 48843,)","(26TH ST PROPERTIES LLC,)",2
38,39,3030 S 9TH LLC,"PO BOX 312098, NEW BRAUNFELS, TX 78131",True,"(PO BOX 312098, NEW BRAUNFELS, TX 78131,)","(3030 S 9TH LLC,)",2
41,42,3381 FOREST VIEW INVESTORS LLC,"2265 LIVERNOIS STE 500, TROY, MI 48083",True,"(2265 LIVERNOIS STE 500, TROY, MI 48083,)","(3381 FOREST VIEW INVESTORS LLC,)",4
42,43,"3501 LAKE HOLDINGS, LLC","3501 LAKE ST, KALAMAZOO, MI 49048",True,"(3501 LAKE ST, KALAMAZOO, MI 49048,)","(3501 LAKE HOLDINGS, LLC,)",2
49,50,5752 TRUST,"4426 HUGH HOWELL RD # B-369, TUCKER, GA 30084",True,"(4426 HUGH HOWELL RD # B-369, TUCKER, GA 30084,)","(5752 TRUST,)",2


There are 1,580 cases of entities that used to have a blank address
- **QUESTION: Is the main thing to look for here that the source_names and canonical_name are in fact similar enough that we would want to merge them?**

## Names affected by CORP/CORPORATION, INC/INCORPORATED, CO/COMPANY

In [7]:
ENTITY_NAME_REPLACEMENTS = {
    "CORP": "CORPORATION",
    "INC": "INCORPORATED",
    "CO": "COMPANY"
}


def expand_entity_abbreviations(name):

    if pd.isna(name):
        return ""

    name = str(name).upper().strip()

    name = name.replace(".", "")

    name = re.sub(r"\s+", " ", name)
    name = re.sub(r"\s*,\s*", ", ", name)

    for old, new in ENTITY_NAME_REPLACEMENTS.items():

        name = re.sub(
            rf"\b{re.escape(old)}\b",
            new,
            name
        )

    return name.strip()

abbreviation_audit = resolution_audit.copy()

abbreviation_audit["expanded_name"] = (
    abbreviation_audit[
        "entity_name"
    ]
    .apply(expand_entity_abbreviations)
)

abbreviation_cases = (
    abbreviation_audit
    .groupby(
        [
            "entity_id",
            "canonical_name",
            "canonical_address"
        ]
    )
    .agg(
        source_names=(
            "entity_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        expanded_names=(
            "expanded_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        source_name_count=(
            "entity_name",
            "nunique"
        ),
        expanded_name_count=(
            "expanded_name",
            "nunique"
        )
    )
    .reset_index()
)

abbreviation_cases = (
    abbreviation_cases[
        (abbreviation_cases["source_name_count"] > 1)
        &
        (abbreviation_cases["expanded_name_count"] == 1)
    ]
)

abbreviation_cases.head(50)

,entity_id,canonical_name,canonical_address,source_names,expanded_names,source_name_count,expanded_name_count
111,112,AIM CORPORATION,"218 GLADYS ST, PORTAGE, MI 49002","(AIM CORP, AIM CORPORATION)","(AIM CORPORATION,)",2,1
2016,2017,J. SAJE LLC,"2900 KENSINGTON DR, KALAMAZOO, MI 49008","(J SAJE LLC, J. SAJE LLC)","(J SAJE LLC,)",2,1
2568,2569,MAIN QSR INC.,"1946 45TH ST STE C, MUNSTER, IN 46321","(MAIN QSR INC, MAIN QSR INC.)","(MAIN QSR INCORPORATED,)",2,1
3122,3123,"ONE DAY, LLC.","1417 ACADEMY, KALAMAZOO, MI 49006","(ONE DAY, LLC, ONE DAY, LLC.)","(ONE DAY, LLC,)",2,1
3667,3668,"S.T.D. MANAGEMENT, LLC","3001 10TH ST, WAYLAND, MI 49348","(S.T.D. MANAGEMENT, LLC, STD MANAGEMENT, LLC)","(STD MANAGEMENT, LLC,)",2,1


- There is 1 case of a CORP/CORPORATION merger.
- Other than that, the other 5 cases involve merging source names that differ by the inclusion/exclusion of a period.
    - For example, ONE DAY, LLC and ONE DAY, LLC.

## Suffix cases

### All people containing a suffix

In [8]:
suffix_pattern = (
    r"\b(?:JR|SR|JUNIOR|SENIOR|II|III|IV)\b"
)

suffix_entities = (
    entities[
        (entities["entity_type"] == "Person")
        &
        (
            entities["canonical_name"]
            .str.contains(
                suffix_pattern,
                regex=True,
                na=False
            )
        )
    ]
    .sort_values(
        "occurrence_count",
        ascending=False
    )
)

print(suffix_entities.shape)

suffix_entities[
    [
        "entity_id",
        "canonical_name",
        "canonical_address",
        "occurrence_count"
    ]
].head(50)

(106, 7)


,entity_id,canonical_name,canonical_address,occurrence_count
167,168,"ANDERSON, A DAVID SR","PO BOX 50204, KALAMAZOO, MI 49005-0204",15
3390,3391,"PRIMER, DANIEL L JR","6841 N SPRINKLE RD, KALAMAZOO, MI 49004",12
2284,2285,"KNOX, DANNY LEE SR","6336 RAVINE RD, KALAMAZOO, MI 49009-9002",9
2035,2036,"JACKSON, SOL JR","5370 LUCERNE AVE, KALAMAZOO, MI 49048",8
1174,1175,"DERRING III, ALBERT","3041 N 26TH ST, KALAMAZOO, MI 49048",7
1110,1111,"DAVIS, KEITH SR.","3850 ROLLRIDGE, KALAMAZOO, MI 49004",5
1626,1627,"GOOCH, MYRON EVERETT JR","1226 CADET LN, KALAMAZOO, MI 49009",4
631,632,"BROWN, ROBERT W. JR.","1115 DUPONT, KALAMAZOO, MI 49048-3213",4
632,633,"BROWN, ROBERT WENDELL, JR.","1115 DUPONT AVE, KALAMAZOO, MI 49048-3213",4
237,238,AUSTIN ART JR,"10595 TOMKINSON DR, SCOTTS, MI 49088",4


- 146 entities contain suffixes

### Entities with both suffix and suffix-free versions

In [9]:
suffix_audit = (
    resolution_audit[
        resolution_audit[
            "entity_type_final"
        ] == "Person"
    ]
    .copy()
)

suffix_audit["has_suffix"] = (
    suffix_audit[
        "entity_name"
    ]
    .str.contains(
        suffix_pattern,
        regex=True,
        na=False
    )
)

suffix_merge_cases = (
    suffix_audit
    .groupby(
        [
            "entity_id",
            "canonical_name",
            "canonical_address"
        ]
    )
    .agg(
        source_names=(
            "entity_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        has_suffix_version=(
            "has_suffix",
            "any"
        ),
        has_no_suffix_version=(
            "has_suffix",
            lambda x: (~x).any()
        )
    )
    .reset_index()
)

suffix_merge_cases = (
    suffix_merge_cases[
        suffix_merge_cases["has_suffix_version"]
        &
        suffix_merge_cases["has_no_suffix_version"]
    ]
)

suffix_merge_cases.head(50)

,entity_id,canonical_name,canonical_address,source_names,has_suffix_version,has_no_suffix_version
788,1175,"DERRING III, ALBERT","3041 N 26TH ST, KALAMAZOO, MI 49048","(DERRING ALBERT III, DERRING III, ALBERT, DERR...",True,True


- There is 1 case of an entity merged due to having the same name and address, with different suffixes
    - Albert Derring and Albert Derring III

## Cases of finalized entities with the same name but different associated addresses

In [10]:
person_entities = (
    entities[
        entities["entity_type"] == "Person"
    ]
    .copy()
)

known_address_people = (
    person_entities[
        person_entities[
            "canonical_address"
        ].fillna("").str.strip() != ""
    ]
)

common_names_multiple_addresses = (
    known_address_people
    .groupby("canonical_name")
    .agg(
        entity_count=(
            "entity_id",
            "nunique"
        ),
        address_count=(
            "canonical_address",
            "nunique"
        ),
        addresses=(
            "canonical_address",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        entity_ids=(
            "entity_id",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        total_occurrences=(
            "occurrence_count",
            "sum"
        )
    )
    .reset_index()
)

common_names_multiple_addresses = (
    common_names_multiple_addresses[
        common_names_multiple_addresses[
            "address_count"
        ] > 1
    ]
    .sort_values(
        [
            "address_count",
            "total_occurrences"
        ],
        ascending=False
    )
)

print(common_names_multiple_addresses.shape)
common_names_multiple_addresses.head(50)

(267, 6)


,canonical_name,entity_count,address_count,addresses,entity_ids,total_occurrences
1922,"OWENS, LESSIE T",5,5,"(1214 N CHURCH ST, KALAMAZOO, MI 49007, 1223 N...","(3160, 3161, 3162, 3163, 3164)",7
933,"GATES, JEROME",4,4,"(1229 ORCHID AVE, SAINT LOUIS, MO 63147, 1229 ...","(1540, 1541, 1542, 1543)",18
2502,"TARDANI, JACOB A",4,4,"(1011 DENNER ST UNIT 1, KALAMAZOO, MI 49006, 1...","(4101, 4102, 4103, 4104)",7
152,"BALL, RONALD",4,4,"(1004 WOODWARD AVE, KALAMAZOO, MI 49007, 1821 ...","(299, 300, 301, 302)",5
1833,NEMIRE JAMES J,4,4,"(7167 SAGINAW DR, COMSTOCK, MI 49041, 7167 SAG...","(3004, 3005, 3006, 3007)",5
2261,"SANTIAGO, JORGE",3,3,"(5101 SWALLOW AVE, KALAMAZOO, MI 49009, 5350 G...","(3718, 3719, 3720)",15
2262,"SANTIAGO, MARIA",3,3,"(5101 SWALLOW AVE, KALAMAZOO, MI 49009, 5350 G...","(3724, 3725, 3726)",15
323,"BRINNEMAN, MICHAEL J",3,3,"(1802 FULFORD ST, KALAMAZOO, MI 49001, 5037 ST...","(578, 579, 580)",11
2325,"SHERROD, ANTHONY",3,3,"(1523 N WESTNEDGE AVE, KALAMAZOO, MI 49007, 36...","(3812, 3813, 3814)",8
1359,"KERNELL, BRIANT",3,3,"(10238 PORTAGE RD, PORTAGE, MI 49002, 10238 PO...","(2227, 2228, 2229)",7


- There are 296 canonical_names which are associated with more than one entity/address

## Inspecting name formatting

In [11]:
people = (
    entity_occurrences[
        entity_occurrences["entity_type"] == "Person"
    ]
)

comma_names = (
    people["entity_name"]
    .str.contains(",", regex=False, na=False)
)

print(
    f"Person occurrences with comma-format names: "
    f"{comma_names.sum():,}"
)

print(
    f"Person occurrences without comma-format names: "
    f"{(~comma_names).sum():,}"
)

print(
    f"Percent parseable by current comma requirement: "
    f"{comma_names.mean():.1%}"
)

Person occurrences with comma-format names: 5,904
Person occurrences without comma-format names: 1,593
Percent parseable by current comma requirement: 78.8%


## Inspecting unknown entity types

In [12]:
entity_occurrences[
    "entity_type"
].value_counts()

entity_type
Person          7497
Organization    1975
Joint           1518
Unknown           22
Name: count, dtype: int64

In [13]:
unknown_names = (
    entity_occurrences.loc[
        entity_occurrences["entity_type"] == "Unknown",
        "entity_name"
    ]
    .value_counts()
    .rename_axis("entity_name")
    .reset_index(name="occurrence_count")
)

print(
    f"Unique Unknown names: "
    f"{len(unknown_names):,}"
)

unknown_names.head(100)

Unique Unknown names: 12


,entity_name,occurrence_count
0,PAYMENT IN LIEU OF TAXES,4
1,"MCNEES, JR",3
2,"MI 4004 DOUGLAS AVE. KALAMAZOO, LL",2
3,CENTURYLINK,2
4,TRSTEE,2
5,RYAN,2
6,AIM,2
7,BENCHMARK,1
8,COLLABORATIVE,1
9,PO BOX 183,1


In [14]:
joint_names = (
    entity_occurrences.loc[
        entity_occurrences[
            "entity_type"
        ] == "Joint",
        "entity_name"
    ]
    .drop_duplicates()
)

joint_names.sample(
    min(100, len(joint_names)),
    random_state=42
).sort_values()

4353         AARON DANIEL & AABERG GAYE
116                  AHLRICH JANITH L &
414         ALDERMAN JONATHAN & REBEKAH
285     AMBRO ANDREW J & MARILEE K (LE)
166                  AMES TERRY & SARAH
                     ...               
3451        WHITE THEODORE A & MARGARET
249     WINCHESTER, CANDACE & GREGORY A
6707          WITT DAVID JC & MENDY RAE
3723             WYMAN ASHLEY & MATTHEW
2895            YERDEN, GARY H & MARY A
Name: entity_name, Length: 100, dtype: str

In [15]:
joint_summary = pd.Series({
    "Contains &": joint_names.str.contains(
        "&",
        regex=False
    ).sum(),

    "Contains AND": joint_names.str.contains(
        r"\bAND\b",
        regex=True
    ).sum(),

    "Contains /": joint_names.str.contains(
        "/",
        regex=False
    ).sum(),

    "Contains comma": joint_names.str.contains(
        ",",
        regex=False
    ).sum(),
})

joint_summary

Contains &        625
Contains AND        7
Contains /         26
Contains comma    249
dtype: int64

## Inspecting Joint type entities

In [16]:
joint_names = (
    entity_occurrences.loc[
        entity_occurrences["entity_type"] == "Joint",
        "entity_name"
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

def profile_joint_name(name):

    candidate = resolver._normalize_entity_type_candidate(name)

    return pd.Series({
        "ampersand_count": candidate.count("&"),
        "semicolon_count": candidate.count(";"),
        "slash_count": candidate.count("/"),

        "and_count": len(
            re.findall(
                r"\bAND\b",
                candidate
            )
        ),

        "comma_count": candidate.count(","),

        "contains_et_al": bool(
            re.search(
                r"\bET AL\b",
                candidate
            )
        ),

        "token_count": len(
            candidate.split()
        )
    })

joint_profile = (
    joint_names
    .to_frame(name="entity_name")
)

joint_profile = pd.concat(
    [
        joint_profile,

        joint_profile["entity_name"]
        .apply(profile_joint_name)
    ],
    axis=1
)

joint_profile.head()

,entity_name,ampersand_count,semicolon_count,slash_count,and_count,comma_count,contains_et_al,token_count
0,CLAWSON DORRINDA &,1,0,0,0,0,False,3
1,SPIEKERMANN TIM & JUDY,1,0,0,0,0,False,4
2,LINDBERG MARK H & LYNNETTE,1,0,0,0,0,False,5
3,THOMSON STEVEN L & ASHLEY A,1,0,0,0,0,False,6
4,MCFALDA JOHN J & LUCRECIE A,1,0,0,0,0,False,6


In [17]:
joint_profile["separator_type"] = "Other"

joint_profile.loc[
    (
        joint_profile["ampersand_count"] == 1
    )
    &
    (
        joint_profile[
            [
                "semicolon_count",
                "slash_count",
                "and_count"
            ]
        ].sum(axis=1) == 0
    ),
    "separator_type"
] = "Single ampersand"

joint_profile.loc[
    joint_profile["semicolon_count"] > 0,
    "separator_type"
] = "Semicolon"

joint_profile.loc[
    joint_profile["slash_count"] > 0,
    "separator_type"
] = "Slash"

joint_profile.loc[
    joint_profile["and_count"] > 0,
    "separator_type"
] = "AND"

joint_profile.loc[
    (
        joint_profile[
            [
                "ampersand_count",
                "semicolon_count",
                "slash_count",
                "and_count"
            ]
        ].sum(axis=1) > 1
    ),
    "separator_type"
] = "Multiple separators"

joint_profile["separator_type"].value_counts()

separator_type
Single ampersand       597
Multiple separators     31
Slash                   15
AND                      6
Semicolon                3
Name: count, dtype: int64

In [18]:
simple_ampersand = (
    joint_profile[
        (
            joint_profile["separator_type"]
            == "Single ampersand"
        )
        &
        (~joint_profile["contains_et_al"])
    ]
)

print(
    f"Unique simple ampersand Joint names: "
    f"{len(simple_ampersand):,}"
)

simple_ampersand[
    [
        "entity_name",
        "comma_count",
        "token_count"
    ]
].sample(
    min(100, len(simple_ampersand)),
    random_state=42
)

Unique simple ampersand Joint names: 591


,entity_name,comma_count,token_count
552,"RICHTER, GARY P & J F",1,6
646,BROKEMA GARY & MARY C,0,5
212,"LAWSON, E THOMAS & RUTH A",1,6
180,"WILSON, JAMES A & CAROLYN A",1,6
500,WIESSNER JAMES L & PAMELA K,0,6
...,...,...,...
168,"SAUCEDO, MA DELA LUZ & ABERTO JORGE",1,7
165,"TRACY, ELLEN A & RUTH SHEPPARD",1,6
467,ALLGOR TODD & JANIE,0,4
367,FARRELL ANDREANA A & BRANDON,0,5


In [19]:
simple_ampersand["comma_count"].value_counts().sort_index()

comma_count
0    376
1    203
2     11
3      1
Name: count, dtype: int64

In [20]:
simple_ampersand.groupby(
    "comma_count"
)["token_count"].describe()

,count,mean,std,min,25%,50%,75%,max
comma_count,,,,,,,,
0,376.0,4.984043,0.949251,3.0,4.0,5.0,6.0,7.0
1,203.0,5.793103,0.664929,3.0,6.0,6.0,6.0,8.0
2,11.0,5.818182,1.078720,5.0,5.0,5.0,6.5,8.0
3,1.0,8.000000,NaN,8.0,8.0,8.0,8.0,8.0


In [21]:
print(len(simple_ampersand))

print(
    simple_ampersand["comma_count"]
    .isna()
    .sum()
)

591
0


In [22]:
medium_joint_candidates = (
    entity_occurrences[
        (entity_occurrences["entity_type"] == "Joint")
        &
        (
            entity_occurrences["joint_parse_confidence"]
            == "Medium"
        )
    ]
    .copy()
)

print(
    "Medium-confidence Joint occurrences:",
    f"{len(medium_joint_candidates):,}"
)

print(
    "Unique Medium-confidence Joint names:",
    f"{medium_joint_candidates['entity_name'].nunique():,}"
)

Medium-confidence Joint occurrences: 356
Unique Medium-confidence Joint names: 150


In [23]:
medium_joint_candidates[
    "joint_parse_rule"
].value_counts()

joint_parse_rule
shared_surname_comma       340
explicit_surnames_comma     16
Name: count, dtype: int64

In [24]:
medium_joint_names = (
    medium_joint_candidates[
        [
            "entity_name",
            "joint_parse_rule"
        ]
    ]
    .drop_duplicates()
)

medium_joint_names.sample(
    min(100, len(medium_joint_names)),
    random_state=42
).sort_values(
    "entity_name"
)

,entity_name,joint_parse_rule
4012,"BARNETTE, JAMES R & SUE L",shared_surname_comma
525,"BOGARD, ROBERT D. & JENNIFER L.",shared_surname_comma
3987,"BRADSHAW, ROBERT A & CARRIE A",shared_surname_comma
4006,"BRADY, DANIEL T & DAWN G",shared_surname_comma
922,"BROOKS, SAMUEL C & DORCAS A",shared_surname_comma
...,...,...
1176,"WILSON, JAMES A & CAROLYN A",shared_surname_comma
1324,"WITT, PHILLIP C. & RUTH I. DECEASED",shared_surname_comma
1514,"WOLFE, LANCE L & DEBORAH L",shared_surname_comma
253,"WYNKOOP, WALKER M & DEBORAH N",shared_surname_comma


In [25]:
expanded_joint_audit = (
    entity_occurrences[
        entity_occurrences["joint_expanded"]
    ]
    [
        [
            "source_occurrence_id",
            "source_entity_name",
            "entity_name",
            "joint_party_index",
            "joint_party_count",
            "joint_parse_rule",
            "joint_parse_confidence"
        ]
    ]
    .sort_values(
        [
            "source_occurrence_id",
            "joint_party_index"
        ]
    )
)

expanded_joint_audit.head(100)

,source_occurrence_id,source_entity_name,entity_name,joint_party_index,joint_party_count,joint_parse_rule,joint_parse_confidence
179,180,"NANCE, EDENA & MCNEES, BRENDA","NANCE, EDENA",1.0,2.0,explicit_surnames_comma,High
180,180,"NANCE, EDENA & MCNEES, BRENDA","MCNEES, BRENDA",2.0,2.0,explicit_surnames_comma,High
181,181,"DOOLEN, ELLERY & ARNETT, JESSICA","DOOLEN, ELLERY",1.0,2.0,explicit_surnames_comma,High
182,181,"DOOLEN, ELLERY & ARNETT, JESSICA","ARNETT, JESSICA",2.0,2.0,explicit_surnames_comma,High
184,183,"BEILBY, MICHAEL & DEANNA","BEILBY, MICHAEL",1.0,2.0,shared_surname_comma,High
...,...,...,...,...,...,...,...
860,813,"WADE, GEORGE R & JUDITH","WADE, JUDITH",2.0,2.0,shared_surname_comma,High
861,814,"ANDERSON, A DAVID SR & EUGENE","ANDERSON, A DAVID SR",1.0,2.0,shared_surname_comma,High
862,814,"ANDERSON, A DAVID SR & EUGENE","ANDERSON, EUGENE",2.0,2.0,shared_surname_comma,High
874,826,"ANDERSON, A DAVID SR & BERNADETTE","ANDERSON, A DAVID SR",1.0,2.0,shared_surname_comma,High


In [29]:
expanded_joint_pairs = (
    expanded_joint_audit
    .pivot_table(
        index=[
            "source_occurrence_id",
            "source_entity_name",
            "joint_parse_rule",
            "joint_parse_confidence"
        ],
        columns="joint_party_index",
        values="entity_name",
        aggfunc="first"
    )
    .reset_index()
    .rename(
        columns={
            1: "person_1",
            2: "person_2"
        }
    )
)

medium_joint_names.sort_values(
    "entity_name"
).reset_index(
    drop=True
)

,entity_name,joint_parse_rule
0,"APPLEGATE, NEIL E & SHARON K",shared_surname_comma
1,"BARBER, JEREMY M & TAMMY L",shared_surname_comma
2,"BARNETTE, JAMES R & SUE L",shared_surname_comma
3,"BARNIAK, RICHARD J & TAMMERALA L",shared_surname_comma
4,"BAUMGARTNER, JERRY & JAMIE R.",shared_surname_comma
...,...,...
145,"WINCHESTER, CANDACE & GREGORY A",shared_surname_comma
146,"WITT, PHILLIP C. & RUTH I. DECEASED",shared_surname_comma
147,"WOLFE, LANCE L & DEBORAH L",shared_surname_comma
148,"WYNKOOP, WALKER M & DEBORAH N",shared_surname_comma


In [27]:
expanded_counts = (
    entity_occurrences[
        entity_occurrences["joint_expanded"]
    ]
    .groupby(
        "source_occurrence_id"
    )
    .size()
)

print(
    "Expanded source occurrences not producing exactly 2 rows:",
    (expanded_counts != 2).sum()
)

Expanded source occurrences not producing exactly 2 rows: 0


In [28]:
expanded_source_ids = set(
    entity_occurrences.loc[
        entity_occurrences["joint_expanded"],
        "source_occurrence_id"
    ]
)

surviving_joint_rows = (
    entity_occurrences[
        entity_occurrences["source_occurrence_id"]
        .isin(expanded_source_ids)
        &
        (
            entity_occurrences["entity_type"]
            == "Joint"
        )
    ]
)

print(
    "Expanded sources still containing a Joint row:",
    len(surviving_joint_rows)
)

Expanded sources still containing a Joint row: 0
